In [1]:
from statsbombpy import sb
import pandas as pd
pd.options.display.max_columns = None
#pd.options.display.max_rows = None

In [2]:
sb.competitions()

c:\Users\tobia\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093,2025-11-15T23:17:41.827093,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,NaN,NaN,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2026-05-12T21:18:08.827431,2026-05-02T02:07:18.902396,2026-05-02T02:07:18.902396,2026-05-12T21:18:08.827431
3,16,4,Europe,Champions League,male,False,False,2018/2019,2026-05-15T15:54:04.598614,2021-06-13T16:17:31.694,NaN,2026-05-15T15:54:04.598614
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,NaN,2024-02-13T02:35:28.134882
...,...,...,...,...,...,...,...,...,...,...,...,...
75,35,75,Europe,UEFA Europa League,male,False,False,1988/1989,2026-04-11T12:48:10.012987,2021-06-13T16:17:31.694,NaN,2026-04-11T12:48:10.012987
76,53,315,Europe,UEFA Women's Euro,female,False,True,2025,2026-04-27T22:02:42.690507,2026-04-27T22:03:28.087062,2026-04-27T22:03:28.087062,2026-04-27T22:02:42.690507
77,53,106,Europe,UEFA Women's Euro,female,False,True,2022,2026-05-05T03:03:04.199896,2026-05-05T03:05:32.480837,2026-05-05T03:05:32.480837,2026-05-05T03:03:04.199896
78,72,107,International,Women's World Cup,female,False,True,2023,2026-05-03T13:51:31.021141,2026-05-03T13:55:52.303219,2026-05-03T13:55:52.303219,2026-05-03T13:51:31.021141


In [3]:
bl = sb.competition_events(
    country = "Germany",
    division = "1. Bundesliga",
    season = "2023/2024",
    gender = "male")

c:\Users\tobia\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsbombpy\api_client.py:23: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [4]:
bl["match_id"].nunique()

34

In [5]:
sorted(bl['team'].unique())

['Augsburg',
 'Bayer Leverkusen',
 'Bayern Munich',
 'Bochum',
 'Borussia Dortmund',
 'Borussia Mönchengladbach',
 'Darmstadt 98',
 'Eintracht Frankfurt',
 'FC Heidenheim',
 'FC Köln',
 'FSV Mainz 05',
 'Freiburg',
 'Hoffenheim',
 'RB Leipzig',
 'Union Berlin',
 'VfB Stuttgart',
 'Werder Bremen',
 'Wolfsburg']

In [6]:
len(bl)

137765

In [7]:
wirtz_id = 40724
#bl[(bl["type"] == "Carry") & (bl["player_id"] == wirtz_id)]


In [8]:
wirtz = bl[(bl["player_id"] == wirtz_id) | (bl["substitution_replacement_id"] == wirtz_id)]
#wirtz[wirtz["pass_shot_assist"] == True]

In [9]:
matches_played = wirtz.groupby(["team", "player", "player_id"]).agg({"match_id":"nunique", "substitution_replacement":"count"})
matches_played.columns = ["played_matches", "subbed_on_sum"]
matches_played = matches_played.reset_index()
#I check here how many times Wirtz was subbed on
subbed_on = wirtz[wirtz["substitution_replacement_id"] == wirtz_id]["substitution_replacement_id"].count()
#Subbed on 6 times


In [10]:
min_until_subbed = wirtz[["player", "minute"]][wirtz["substitution_replacement"].isna() == False]
play_as_sub = min_until_subbed["minute"].count()
min_until_subbed_off_sum = min_until_subbed[min_until_subbed["player"] == "Florian Wirtz" ]["minute"].sum()
min_playing_after_sub_sum = (90 - min_until_subbed[min_until_subbed["player"] != "Florian Wirtz" ]["minute"]).sum()
full_matches_played = matches_played[matches_played["player_id"] == wirtz_id]["played_matches"].sum() - play_as_sub
min_sum = full_matches_played*90 + min_until_subbed_off_sum + min_playing_after_sub_sum
min_sum
#I know, that a match takes longer than 90 minutes, but it's a simplification, and apparently everyone uses it, because what is the chance, that FotMob has exactly the same number of minutes
#and FBref has only 5 mins less


np.int64(2377)

In [11]:
passes = bl.groupby(["team", "player"]).agg({"pass_length":["count", "sum"],
                                             "pass_outcome":"count",
                                             "pass_through_ball":"count",
                                             "pass_shot_assist": "count",
                                             "pass_goal_assist":"count"})

passes.columns = ["num_of_passes", "total_pass_length", "passes_incompleted", "through_balls", "shot_assists", "assists"]
passes = passes.reset_index()
passes = passes[passes["team"] == "Bayer Leverkusen"]
passes["total_pass_length"] = passes["total_pass_length"].round(2)
passes["passes_completed"] = passes["num_of_passes"] - passes["passes_incompleted"]
passes["pass%"] = round(((passes["passes_completed"] / passes["num_of_passes"]) * 100), 2)
passes["length_per_pass"] = round((passes["total_pass_length"] / passes["num_of_passes"]))
passes["mean_successful_pass_length"] = round(((passes["pass%"]/100)*passes["length_per_pass"]))
passes[passes["num_of_passes"] > 999].sort_values(by = "mean_successful_pass_length", ascending=False).head(11)

,team,player,num_of_passes,total_pass_length,passes_incompleted,through_balls,shot_assists,assists,passes_completed,pass%,length_per_pass,mean_successful_pass_length
27,Bayer Leverkusen,Edmond Fayçal Tapsoba,1946,37395.14,163,7,6,1,1783,91.62,19.0,17.0
30,Bayer Leverkusen,Granit Xhaka,3299,57953.39,254,19,41,0,3045,92.30,18.0,17.0
34,Bayer Leverkusen,Jonathan Tah,2167,38531.43,110,0,4,1,2057,94.92,18.0,17.0
41,Bayer Leverkusen,Odilon Kossonou,1422,25168.49,134,6,11,0,1288,90.58,18.0,16.0
43,Bayer Leverkusen,Piero Martín Hincapié Reyna,1423,24357.17,109,3,4,1,1314,92.34,17.0,16.0
35,Bayer Leverkusen,Josip Stanišić,1072,19005.17,109,3,12,1,963,89.83,18.0,16.0
28,Bayer Leverkusen,Exequiel Alejandro Palacios,2022,31396.03,155,9,23,4,1867,92.33,16.0,15.0
33,Bayer Leverkusen,Jonas Hofmann,1528,26917.23,289,5,69,7,1239,81.09,18.0,15.0
44,Bayer Leverkusen,Robert Andrich,1501,26035.87,142,5,19,2,1359,90.54,17.0,15.0
23,Bayer Leverkusen,Alejandro Grimaldo García,2159,35993.96,376,15,63,13,1783,82.58,17.0,14.0


In [12]:
#I will take only rows with a pass that assisted a shot
every_assist = bl[(bl["pass_assisted_shot_id"].isna() == False) & (bl["team"] == "Bayer Leverkusen")][["player_id", "player", "team", "pass_assisted_shot_id"]]
#I will search only for Leverkusen's players shots, by shot_body_part (but I can take here anything that will have a shot value) and load the xG
every_shot = bl[(bl["team"] == "Bayer Leverkusen") & (bl["type"] == "Shot")][["shot_outcome", "id", "shot_statsbomb_xg"]]
#I merge two tables into one, by id of the shot and use left merge because I want only the shots that Wirtz assisted
merged = pd.merge(
    left = every_assist,
    right = every_shot,
    left_on = "pass_assisted_shot_id",
    right_on = "id",
    how = "left"
)
xA = merged.groupby(["player_id", "player"]).agg({"shot_statsbomb_xg":"sum"})
xA = xA.reset_index()
xA = xA[["player_id", "player", "shot_statsbomb_xg"]].sort_values(by = "shot_statsbomb_xg", ascending = False)
xA
#xA = 7.732809 which seems very low compared to independent data sources

,player_id,player,shot_statsbomb_xg
6,10336.0,Alejandro Grimaldo García,9.147892
4,8804.0,Jonas Hofmann,7.839494
16,40724.0,Florian Wirtz,7.732809
13,32712.0,Jeremie Frimpong,4.543442
12,32289.0,Victor Okoh Boniface,3.831032
10,28268.0,Exequiel Alejandro Palacios,3.334345
0,3500.0,Granit Xhaka,2.605642
14,33401.0,Amine Adli,2.436795
17,41411.0,Nathan Tella,2.331407
5,9195.0,Robert Andrich,1.786079


In [13]:
bl.head(1)

,50_50,bad_behaviour_card,ball_receipt_outcome,ball_recovery_offensive,ball_recovery_recovery_failure,block_deflection,block_offensive,block_save_block,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_other,clearance_right_foot,counterpress,dribble_no_touch,dribble_nutmeg,dribble_outcome,dribble_overrun,duel_outcome,duel_type,duration,foul_committed_advantage,foul_committed_card,foul_committed_offensive,foul_committed_penalty,foul_committed_type,foul_won_advantage,foul_won_defensive,foul_won_penalty,goalkeeper_body_part,goalkeeper_end_location,goalkeeper_outcome,goalkeeper_position,goalkeeper_punched_out,goalkeeper_shot_saved_off_target,goalkeeper_shot_saved_to_post,goalkeeper_success_in_play,goalkeeper_success_out,goalkeeper_technique,goalkeeper_type,id,index,injury_stoppage_in_chain,interception_outcome,location,match_id,minute,miscontrol_aerial_won,off_camera,out,pass_aerial_won,pass_angle,pass_assisted_shot_id,pass_body_part,pass_cross,pass_cut_back,pass_deflected,pass_end_location,pass_goal_assist,pass_height,pass_inswinging,pass_length,pass_miscommunication,pass_no_touch,pass_outcome,pass_outswinging,pass_recipient,pass_recipient_id,pass_shot_assist,pass_straight,pass_switch,pass_technique,pass_through_ball,pass_type,period,play_pattern,player,player_id,position,possession,possession_team,possession_team_id,related_events,second,shot_aerial_won,shot_body_part,shot_deflected,shot_end_location,shot_first_time,shot_freeze_frame,shot_key_pass_id,shot_one_on_one,shot_open_goal,shot_outcome,shot_saved_off_target,shot_saved_to_post,shot_statsbomb_xg,shot_technique,shot_type,substitution_outcome,substitution_outcome_id,substitution_replacement,substitution_replacement_id,tactics,team,team_id,timestamp,type,under_pressure
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49e1fcf3-3fd7-4c72-8d6f-95645e354925,1,NaN,NaN,NaN,3895292,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Regular Play,NaN,NaN,NaN,1,Union Berlin,190,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'formation': 3511, 'lineup': [{'player': {'id...",Union Berlin,190,00:00:00.000,Starting XI,NaN


In [14]:
matches_played = bl[(bl["team"] == "Bayer Leverkusen")].groupby(["team", "player_id", "player"]).agg({"match_id":"nunique", "substitution_replacement":"count"})
matches_played.columns = ["played_matches", "subbed_off_sum"]
matches_played = matches_played.reset_index()

subbed_on = bl[bl["team"] == "Bayer Leverkusen"].groupby("substitution_replacement").agg({"player":"count"})
subbed_on = subbed_on.fillna(value = 0, inplace = True)
subbed_on = subbed_on.reset_index()
subbed_on.columns = ["player", "subbed_on_sum"]

min_until_subbed_off = bl[bl["team"] == "Bayer Leverkusen"][bl["substitution_replacement"].isna() == False].groupby("player").agg({"minute":"sum"})
min_until_subbed_off.columns = ["mins_until_subbed_off"]
min_until_subbed_off = min_until_subbed_off.reset_index()

min_after_subbed_on = bl[(bl["team"] == "Bayer Leverkusen") & (bl["substitution_replacement"].isna() == False)][["substitution_replacement", "minute"]].groupby("substitution_replacement").agg({"minute":"sum"})
min_after_subbed_on["subbed_on_sum"] = bl[bl["team"] == "Bayer Leverkusen"].groupby("substitution_replacement").agg({"substitution_replacement":"count"})
min_after_subbed_on.columns = ["mins_in_progress", "times_subbed_on"]
min_after_subbed_on = min_after_subbed_on.reset_index()
min_after_subbed_on["min_after_subbed_on"] = min_after_subbed_on["times_subbed_on"]*90 - min_after_subbed_on["mins_in_progress"]
min_after_subbed_on = min_after_subbed_on.drop(labels = ["mins_in_progress", "times_subbed_on"], axis = 1)
combined = pd.merge(
    left = matches_played,
    right = subbed_on,
    left_on = "player",
    right_on = "player",
    how = "left")
combined = combined.fillna(value = 0, inplace = True)
combined["matches_fully_played"] = combined["played_matches"] - combined["subbed_off_sum"] - combined["subbed_on_sum"]

combined["full_matches_min"] = combined["matches_fully_played"]*90
combined = pd.merge(
    left = combined,
    right = min_until_subbed_off,
    left_on = "player",
    right_on = "player",
    how = "left")
combined = combined.fillna(value = 0, inplace = True)
combined = pd.merge(
    left = combined,
    right = min_after_subbed_on,
    left_on = "player",
    right_on = "substitution_replacement",
    how = "left")
combined = combined.drop(labels = ["team", "substitution_replacement"], axis =1)
combined = combined.fillna(value = 0, inplace = True)
combined["total_min"] = combined["full_matches_min"] + combined["mins_until_subbed_off"] + combined["min_after_subbed_on"]
combined["per90"] = round(combined["total_min"]/90, 1)
per_90 = combined.drop(labels = ["played_matches", "subbed_off_sum", "subbed_on_sum", "matches_fully_played", "full_matches_min", "mins_until_subbed_off", "min_after_subbed_on"], axis = 1)
per_90

C:\Users\tobia\AppData\Local\Temp\ipykernel_19012\497846822.py:10: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  min_until_subbed_off = bl[bl["team"] == "Bayer Leverkusen"][bl["substitution_replacement"].isna() == False].groupby("player").agg({"minute":"sum"})


,player_id,player,total_min,per90
0,3500.0,Granit Xhaka,2821.0,31.3
1,7044.0,Patrik Schick,1062.0,11.8
2,8221.0,Jonathan Tah,2631.0,29.2
3,8403.0,Nadiem Amiri,81.0,0.9
4,8667.0,Lukáš Hrádecký,2970.0,33.0
5,8804.0,Jonas Hofmann,2205.0,24.5
6,9195.0,Robert Andrich,1689.0,18.8
7,10336.0,Alejandro Grimaldo García,2785.0,30.9
8,11391.0,Borja Iglesias Quintas,255.0,2.8
9,27133.0,Odilon Kossonou,1811.0,20.1


In [15]:
through_balls = bl[(bl["pass_technique"] == "Through Ball") & (bl["team"] == "Bayer Leverkusen")][["player", "player_id", "pass_technique"]].groupby(["player_id"]).agg({"pass_technique":"count"})
through_balls.columns = ["Through Balls"]
through_balls = through_balls.reset_index()
through_balls_per90 = pd.merge(
    left = per_90,
    right = through_balls,
    left_on = "player_id",
    right_on = "player_id",
    how = "right")
through_balls_per90["Through Balls per 90"] = round(through_balls_per90["Through Balls"] / through_balls_per90["per90"], 2)
through_balls_per90[through_balls_per90["total_min"] > 999].sort_values(by = "Through Balls per 90", ascending = False)

,player_id,player,total_min,per90,Through Balls,Through Balls per 90
12,40724.0,Florian Wirtz,2377.0,26.4,22,0.83
0,3500.0,Granit Xhaka,2821.0,31.3,19,0.61
4,10336.0,Alejandro Grimaldo García,2785.0,30.9,15,0.49
6,28268.0,Exequiel Alejandro Palacios,1832.0,20.4,9,0.44
8,32289.0,Victor Okoh Boniface,1545.0,17.2,7,0.41
7,30606.0,Edmond Fayçal Tapsoba,1997.0,22.2,7,0.32
5,27133.0,Odilon Kossonou,1811.0,20.1,6,0.30
9,32712.0,Jeremie Frimpong,2247.0,25.0,7,0.28
3,9195.0,Robert Andrich,1689.0,18.8,5,0.27
14,49337.0,Josip Stanišić,1262.0,14.0,3,0.21


In [16]:
xA_per90 = pd.merge(
    left = xA,
    right = per_90,
    left_on = "player_id",
    right_on = "player_id",
    how = "left")
xA_per90 = xA_per90.drop(labels = "player_y", axis = 1)
xA_per90.columns = ["player_id", "player", "xA", "total_min", "per90"]
xA_per90["xA_per90"] = round(xA_per90["xA"]/xA_per90["per90"], 2)
xA_per90[xA_per90["total_min"] > 999].sort_values(by = "xA_per90", ascending = False)

,player_id,player,xA,total_min,per90,xA_per90
1,8804.0,Jonas Hofmann,7.839494,2205.0,24.5,0.32
0,10336.0,Alejandro Grimaldo García,9.147892,2785.0,30.9,0.30
2,40724.0,Florian Wirtz,7.732809,2377.0,26.4,0.29
4,32289.0,Victor Okoh Boniface,3.831032,1545.0,17.2,0.22
3,32712.0,Jeremie Frimpong,4.543442,2247.0,25.0,0.18
5,28268.0,Exequiel Alejandro Palacios,3.334345,1832.0,20.4,0.16
10,49337.0,Josip Stanišić,1.690860,1262.0,14.0,0.12
9,9195.0,Robert Andrich,1.786079,1689.0,18.8,0.10
6,3500.0,Granit Xhaka,2.605642,2821.0,31.3,0.08
13,7044.0,Patrik Schick,0.830989,1062.0,11.8,0.07


In [17]:
bl[(bl["pass_technique"] == "Through Ball") & (bl["team"] == "Bayer Leverkusen")].groupby("play_pattern").agg({"pass_through_ball":"count"})

,pass_through_ball
play_pattern,
From Corner,9
From Counter,6
From Free Kick,10
From Goal Kick,8
From Keeper,7
From Kick Off,1
From Throw In,21
Regular Play,53


In [18]:
bl[(bl["pass_technique"] == "Through Ball") & (bl["team"] == "Bayer Leverkusen")].head(5)

,50_50,bad_behaviour_card,ball_receipt_outcome,ball_recovery_offensive,ball_recovery_recovery_failure,block_deflection,block_offensive,block_save_block,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_other,clearance_right_foot,counterpress,dribble_no_touch,dribble_nutmeg,dribble_outcome,dribble_overrun,duel_outcome,duel_type,duration,foul_committed_advantage,foul_committed_card,foul_committed_offensive,foul_committed_penalty,foul_committed_type,foul_won_advantage,foul_won_defensive,foul_won_penalty,goalkeeper_body_part,goalkeeper_end_location,goalkeeper_outcome,goalkeeper_position,goalkeeper_punched_out,goalkeeper_shot_saved_off_target,goalkeeper_shot_saved_to_post,goalkeeper_success_in_play,goalkeeper_success_out,goalkeeper_technique,goalkeeper_type,id,index,injury_stoppage_in_chain,interception_outcome,location,match_id,minute,miscontrol_aerial_won,off_camera,out,pass_aerial_won,pass_angle,pass_assisted_shot_id,pass_body_part,pass_cross,pass_cut_back,pass_deflected,pass_end_location,pass_goal_assist,pass_height,pass_inswinging,pass_length,pass_miscommunication,pass_no_touch,pass_outcome,pass_outswinging,pass_recipient,pass_recipient_id,pass_shot_assist,pass_straight,pass_switch,pass_technique,pass_through_ball,pass_type,period,play_pattern,player,player_id,position,possession,possession_team,possession_team_id,related_events,second,shot_aerial_won,shot_body_part,shot_deflected,shot_end_location,shot_first_time,shot_freeze_frame,shot_key_pass_id,shot_one_on_one,shot_open_goal,shot_outcome,shot_saved_off_target,shot_saved_to_post,shot_statsbomb_xg,shot_technique,shot_type,substitution_outcome,substitution_outcome_id,substitution_replacement,substitution_replacement_id,tactics,team,team_id,timestamp,type,under_pressure
895,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.324069,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,af41300d-60f9-4060-89c4-7aba73ed0fc7,2412,NaN,NaN,"[94.7, 26.8]",3895292,58,NaN,NaN,NaN,NaN,0.106547,7b0a71e0-997f-4128-82fe-bc896f6c3b93,Right Foot,NaN,NaN,NaN,"[113.4, 28.8]",NaN,Ground Pass,NaN,18.806648,NaN,NaN,NaN,NaN,Amine Adli,33401.0,True,NaN,NaN,Through Ball,True,NaN,2,Regular Play,Florian Wirtz,40724.0,Right Attacking Midfield,107,Bayer Leverkusen,904,"[5b5125c3-70be-4be6-a96c-9dfbcbe24c58, 63d852d...",28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bayer Leverkusen,904,00:13:28.772,Pass,True
1270,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.948353,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,d131c5fb-add6-4835-b647-abb153df7f54,3788,NaN,NaN,"[84.3, 25.0]",3895292,94,NaN,NaN,NaN,NaN,0.271974,2fda9d56-0749-4c28-8423-7c45bdab5d5c,Right Foot,NaN,NaN,NaN,"[109.4, 32.0]",NaN,Ground Pass,NaN,26.057821,NaN,NaN,NaN,NaN,Amine Adli,33401.0,True,NaN,NaN,Through Ball,True,NaN,2,From Corner,Robert Andrich,9195.0,Left Defensive Midfield,161,Bayer Leverkusen,904,[01a50df7-0f08-423b-b30c-2de7f3a0517a],35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bayer Leverkusen,904,00:49:35.476,Pass,NaN
2396,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.778224,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9a3dfec7-159f-4639-930c-e5022887be4c,131,NaN,NaN,"[66.1, 3.1]",3895158,2,NaN,NaN,NaN,NaN,0.382062,NaN,Right Foot,NaN,NaN,NaN,"[110.4, 20.9]",NaN,High Pass,NaN,47.742330,NaN,NaN,Pass Offside,NaN,Jonas Hofmann,8804.0,NaN,NaN,NaN,Through Ball,True,NaN,1,Regular Play,Florian Wirtz,40724.0,Left Attacking Midfield,7,Bayer Leverkusen,904,[fda8309d-274e-43ca-8f0b-eb79b50a28cf],59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bayer Leverkusen,904,00:02:59.813,Pass,NaN
2421,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2

In [19]:
x_column = bl[bl["pass_technique"] == "Through Ball"]["location"].apply(lambda x: x[0])
x_column

895      94.7
1270     84.3
1871     35.8
2198     70.4
2396     66.1
         ... 
36889    76.9
37810    83.2
38562    89.0
38658    85.6
39165    90.3
Name: location, Length: 180, dtype: float64

In [20]:
tb_from_counter = bl[(bl["play_pattern"] == "From Counter") & (bl["pass_technique"] == "Through Ball") & (bl["team"] == "Bayer Leverkusen")].groupby(["player_id", "player"]).agg({"pass_technique":"count"})
tb_from_counter.columns = ["Through balls from counter"]
tb_from_counter = tb_from_counter.reset_index()
tb_from_buildup = bl[(bl["play_pattern"].isin(["Regular Play", "From Goal Kick", "From Keeper", "From Throw In"])) & (bl["pass_technique"] == "Through Ball") & (bl["team"] == "Bayer Leverkusen")].groupby(["player_id", "player"]).agg({"pass_technique":"count"})
tb_from_buildup.columns = ["Through ball from build-up"]
tb_from_buildup = tb_from_buildup.reset_index()
comparison = pd.merge(
    left = tb_from_counter,
    right = tb_from_buildup,
    left_on = "player_id",
    right_on  = "player_id",
    how = "outer")
comparison = comparison.drop(labels = "player_x", axis=1)
comparison = comparison.loc[:, ["player_id", "player_y", "Through balls from counter", "Through ball from build-up"]]
comparison.columns = ["player_id", "player", "Through balls from counter", "Through balls from build-up"]
comparison = comparison.fillna(value = 0, inplace = True)
comparison = pd.merge(
    left = comparison,
    right = per_90,
    left_on = "player_id",
    right_on = "player_id",
    how = "left")
comparison = comparison.drop(labels = ["player_y", "total_min"], axis=1)
comparison["TB from counter/90"] = round(comparison["Through balls from counter"]/comparison["per90"], 3)
comparison["TB from build-up/90"] = round(comparison["Through balls from build-up"]/comparison["per90"], 3)
comparison = comparison.drop(labels = "per90", axis = 1)
comparison = comparison.loc[:, ["player_id", "player_x", "Through balls from counter", "TB from counter/90", "Through balls from build-up", "TB from build-up/90"]]
comparison.sort_values(by = "TB from build-up/90", ascending = False)


,player_id,player_x,Through balls from counter,TB from counter/90,Through balls from build-up,TB from build-up/90
12,40724.0,Florian Wirtz,1.0,0.038,19,0.720
0,3500.0,Granit Xhaka,0.0,0.000,14,0.447
4,10336.0,Alejandro Grimaldo García,0.0,0.000,13,0.421
8,32289.0,Victor Okoh Boniface,0.0,0.000,7,0.407
10,33401.0,Amine Adli,0.0,0.000,3,0.300
13,41411.0,Nathan Tella,0.0,0.000,2,0.220
14,49337.0,Josip Stanišić,0.0,0.000,3,0.214
3,9195.0,Robert Andrich,0.0,0.000,4,0.213
5,27133.0,Odilon Kossonou,1.0,0.050,4,0.199
6,28268.0,Exequiel Alejandro Palacios,1.0,0.049,4,0.196


In [21]:
wirtz["type"].unique()

<StringArray>
[           'Pass',   'Ball Receipt*',           'Carry',        'Pressure',
   'Ball Recovery',      'Miscontrol',           'Block',   'Dribbled Past',
         'Dribble',    'Dispossessed',            'Duel',       'Clearance',
            'Shot',  'Foul Committed',        'Foul Won',    'Interception',
 'Injury Stoppage',           '50/50',    'Substitution',      'Player Off',
       'Player On',   'Bad Behaviour']
Length: 22, dtype: str

In [22]:
bl.head(1)

,50_50,bad_behaviour_card,ball_receipt_outcome,ball_recovery_offensive,ball_recovery_recovery_failure,block_deflection,block_offensive,block_save_block,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_other,clearance_right_foot,counterpress,dribble_no_touch,dribble_nutmeg,dribble_outcome,dribble_overrun,duel_outcome,duel_type,duration,foul_committed_advantage,foul_committed_card,foul_committed_offensive,foul_committed_penalty,foul_committed_type,foul_won_advantage,foul_won_defensive,foul_won_penalty,goalkeeper_body_part,goalkeeper_end_location,goalkeeper_outcome,goalkeeper_position,goalkeeper_punched_out,goalkeeper_shot_saved_off_target,goalkeeper_shot_saved_to_post,goalkeeper_success_in_play,goalkeeper_success_out,goalkeeper_technique,goalkeeper_type,id,index,injury_stoppage_in_chain,interception_outcome,location,match_id,minute,miscontrol_aerial_won,off_camera,out,pass_aerial_won,pass_angle,pass_assisted_shot_id,pass_body_part,pass_cross,pass_cut_back,pass_deflected,pass_end_location,pass_goal_assist,pass_height,pass_inswinging,pass_length,pass_miscommunication,pass_no_touch,pass_outcome,pass_outswinging,pass_recipient,pass_recipient_id,pass_shot_assist,pass_straight,pass_switch,pass_technique,pass_through_ball,pass_type,period,play_pattern,player,player_id,position,possession,possession_team,possession_team_id,related_events,second,shot_aerial_won,shot_body_part,shot_deflected,shot_end_location,shot_first_time,shot_freeze_frame,shot_key_pass_id,shot_one_on_one,shot_open_goal,shot_outcome,shot_saved_off_target,shot_saved_to_post,shot_statsbomb_xg,shot_technique,shot_type,substitution_outcome,substitution_outcome_id,substitution_replacement,substitution_replacement_id,tactics,team,team_id,timestamp,type,under_pressure
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49e1fcf3-3fd7-4c72-8d6f-95645e354925,1,NaN,NaN,NaN,3895292,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Regular Play,NaN,NaN,NaN,1,Union Berlin,190,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'formation': 3511, 'lineup': [{'player': {'id...",Union Berlin,190,00:00:00.000,Starting XI,NaN


In [23]:
wirtz[wirtz["foul_won_penalty"] == True]

,50_50,bad_behaviour_card,ball_receipt_outcome,ball_recovery_offensive,ball_recovery_recovery_failure,block_deflection,block_offensive,block_save_block,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_other,clearance_right_foot,counterpress,dribble_no_touch,dribble_nutmeg,dribble_outcome,dribble_overrun,duel_outcome,duel_type,duration,foul_committed_advantage,foul_committed_card,foul_committed_offensive,foul_committed_penalty,foul_committed_type,foul_won_advantage,foul_won_defensive,foul_won_penalty,goalkeeper_body_part,goalkeeper_end_location,goalkeeper_outcome,goalkeeper_position,goalkeeper_punched_out,goalkeeper_shot_saved_off_target,goalkeeper_shot_saved_to_post,goalkeeper_success_in_play,goalkeeper_success_out,goalkeeper_technique,goalkeeper_type,id,index,injury_stoppage_in_chain,interception_outcome,location,match_id,minute,miscontrol_aerial_won,off_camera,out,pass_aerial_won,pass_angle,pass_assisted_shot_id,pass_body_part,pass_cross,pass_cut_back,pass_deflected,pass_end_location,pass_goal_assist,pass_height,pass_inswinging,pass_length,pass_miscommunication,pass_no_touch,pass_outcome,pass_outswinging,pass_recipient,pass_recipient_id,pass_shot_assist,pass_straight,pass_switch,pass_technique,pass_through_ball,pass_type,period,play_pattern,player,player_id,position,possession,possession_team,possession_team_id,related_events,second,shot_aerial_won,shot_body_part,shot_deflected,shot_end_location,shot_first_time,shot_freeze_frame,shot_key_pass_id,shot_one_on_one,shot_open_goal,shot_outcome,shot_saved_off_target,shot_saved_to_post,shot_statsbomb_xg,shot_technique,shot_type,substitution_outcome,substitution_outcome_id,substitution_replacement,substitution_replacement_id,tactics,team,team_id,timestamp,type,under_pressure


In [24]:
wirtz_actions = wirtz[(wirtz["type"].isin(values = ["Dispossessed", "Shot", "Foul Won", "Pass", "Miscontrol", "Injury Stoppage", "Error", "Duel", "Dribble"])) 
      & ((wirtz["duel_outcome"] != "Won") & (wirtz["dribble_outcome"] != "Complete"))]["type"].count()
wirtz_actions

np.int64(2235)

In [25]:
wirtz[wirtz["type"] == "Pass"].count()

50_50                                0
bad_behaviour_card                   0
ball_receipt_outcome                 0
ball_recovery_offensive              0
ball_recovery_recovery_failure       0
                                  ... 
team                              1841
team_id                           1841
timestamp                         1841
type                              1841
under_pressure                     352
Length: 111, dtype: int64

In [26]:
wirtz_dribbles = wirtz[wirtz["type"] == "Dribble"]["type"].count()
wirtz_succ_dribbles = wirtz[wirtz["dribble_outcome"] == "Complete"]["dribble_outcome"].count()
dribble_per_action = round(wirtz_dribbles/wirtz_actions, 3)
wirtz_per90 = per_90[per_90["player"] == "Florian Wirtz"]["per90"]
dribbles_per90 = round(wirtz_dribbles/wirtz_per90, 3)
round(wirtz_succ_dribbles/wirtz_dribbles, 2)

np.float64(0.57)